# Revolut case study

### Import packages

In [1]:
import numpy as np
import pandas as pd
import ast # for converting data to dict

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

C:\Users\pitla\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# Import files
doc_report = pd.read_csv('doc_report.csv')
facial_report = pd.read_csv('facial_similarity_report.csv')

### Exploratory data analysis

In [3]:
# View the data 
doc_report.head()

,number,user_id,result,visual_authenticity_result,image_integrity_result,face_detection_result,image_quality_result,created_at,supported_document_result,conclusive_document_quality_result,colour_picture_result,data_validation_result,data_consistency_result,data_comparison_result,attempt_id,police_record_result,compromised_document_result,properties,sub_result
0,0,ab23fae164e34af0a1ad1423ce9fd9f0,consider,consider,clear,clear,clear,2017-06-20T23:12:57Z,clear,NaN,NaN,clear,clear,NaN,050a0596de424fab83c433eaa18b3f8d,clear,NaN,"{'gender': 'Male', 'nationality': 'IRL', 'docu...",caution
1,1,15a84e8951254011b47412fa4e8f65b8,clear,clear,clear,clear,clear,2017-06-20T23:16:04Z,clear,NaN,NaN,clear,NaN,NaN,f69c1e5f45a64e50a26740b9bfb978b7,clear,NaN,"{'gender': 'Female', 'document_type': 'driving...",clear
2,2,ffb82fda52b041e4b9af9cb4ef298c85,clear,clear,clear,clear,clear,2017-06-20T17:59:49Z,clear,NaN,NaN,clear,clear,NaN,f9f84f3055714d8e8f7419dc984d1769,clear,NaN,"{'gender': 'Male', 'nationality': 'ITA', 'docu...",clear
3,3,bd4a8b3e3601427e88aa1d9eab9f4290,clear,clear,clear,clear,clear,2017-06-20T17:59:38Z,clear,NaN,NaN,clear,clear,NaN,10a54a1ecf794404be959e030f11fef6,clear,NaN,"{'gender': 'Male', 'issuing_date': '2007-08', ...",clear
4,4,f52ad1c7e69543a9940c3e7f8ed28a39,clear,clear,clear,clear,clear,2017-06-20T18:08:09Z,clear,NaN,NaN,clear,clear,NaN,1f320d1d07de493292b7e0d5ebfb1cb9,clear,NaN,"{'gender': 'Male', 'nationality': 'POL', 'docu...",clear


### Initial findings: 
- created_date column containing timestamp 
- attempt_id in the middle of the table 
- properties column containing dict like data

In [4]:
# Gathering info on the dataset 
doc_report.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 176404 entries, 0 to 176403
Data columns (total 19 columns):
 #   Column                              Non-Null Count   Dtype 
---  ------                              --------------   ----- 
 0   number                              176404 non-null  int64 
 1   user_id                             176404 non-null  object
 2   result                              176404 non-null  object
 3   visual_authenticity_result          150290 non-null  object
 4   image_integrity_result              176403 non-null  object
 5   face_detection_result               150261 non-null  object
 6   image_quality_result                176403 non-null  object
 7   created_at                          176404 non-null  object
 8   supported_document_result           175900 non-null  object
 9   conclusive_document_quality_result  95217 non-null   object
 10  colour_picture_result               95222 non-null   object
 11  data_validation_result              142

In [5]:
# Convert created_at to date 
doc_report['created_at'] = pd.to_datetime(doc_report['created_at'])

In [6]:
# Create a creation date column
doc_report['created_date'] = doc_report['created_at'].dt.date

### Place ID columns at the beginning

In [7]:
# Move attempt ID to the front of the DF
cols = ['number','user_id','attempt_id'] + [col for col in doc_report.columns if col not in ['number','user_id','attempt_id']]
doc_report = doc_report[cols]

In [8]:
doc_report

,number,user_id,attempt_id,result,visual_authenticity_result,image_integrity_result,face_detection_result,image_quality_result,created_at,supported_document_result,conclusive_document_quality_result,colour_picture_result,data_validation_result,data_consistency_result,data_comparison_result,police_record_result,compromised_document_result,properties,sub_result,created_date
0,0,ab23fae164e34af0a1ad1423ce9fd9f0,050a0596de424fab83c433eaa18b3f8d,consider,consider,clear,clear,clear,2017-06-20 23:12:57+00:00,clear,NaN,NaN,clear,clear,NaN,clear,NaN,"{'gender': 'Male', 'nationality': 'IRL', 'docu...",caution,2017-06-20
1,1,15a84e8951254011b47412fa4e8f65b8,f69c1e5f45a64e50a26740b9bfb978b7,clear,clear,clear,clear,clear,2017-06-20 23:16:04+00:00,clear,NaN,NaN,clear,NaN,NaN,clear,NaN,"{'gender': 'Female', 'document_type': 'driving...",clear,2017-06-20
2,2,ffb82fda52b041e4b9af9cb4ef298c85,f9f84f3055714d8e8f7419dc984d1769,clear,clear,clear,clear,clear,2017-06-20 17:59:49+00:00,clear,NaN,NaN,clear,clear,NaN,clear,NaN,"{'gender': 'Male', 'nationality': 'ITA', 'docu...",clear,2017-06-20
3,3,bd4a8b3e3601427e88aa1d9eab9f4290,10a54a1ecf794404be959e030f11fef6,clear,clear,clear,clear,clear,2017-06-20 17:59:38+00:00,clear,NaN,NaN,clear,clear,NaN,clear,NaN,"{'gender': 'Male', 'issuing_date': '2007-08', ...",clear,2017-06-20
4,4,f52ad1c7e69543a9940c3e7f8ed28a39,1f320d1d07de493292b7e0d5ebfb1cb9,clear,clear,clear,clear,clear,2017-06-20 18:08:09+00:00,clear,NaN,NaN,clear,clear,NaN,clear,NaN,"{'gender': 'Male', 'nationality': 'POL', 'docu...",clear,2017-06-20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176399,181987,c7f99ee763bf45d289019c6ac2cbd919,72a2cccc9e9942deb5274a16536bf2d0,clear,clear,clear,clear,clear,2017-06-20 22:25:53+00:00,clear,NaN,NaN,clear,clear,NaN,clear,NaN,"{'gender': 'Female', 'nationality': 'CHN', 'do...",clear,2017-06-20
176400,181988,8b47d72c363e4591861f523dd7487f20,8c83017191204a3887c2d47ca2d998ce,clear,clear,clear,clear,clear,2017-06-20 22:27:40+00:00,clear,NaN,NaN,clear,NaN,NaN,clear,NaN,"{'gender': 'Female', 'document_type': 'driving...",clear,2017-06-20
176401,181989,3d16e02c245a4f1a8a76662ad933d5c4,bfea35bcb6a940118ca5816cd8ffcae7,clear,clear,clear,clear,clear,2017-06-20 22:25:59+00:00,clear,NaN,NaN,clear,clear,NaN,clear,NaN,"{'gender': 'Female', 'nationality': 'GBR', 'do...",clear,2017-06-20
176402,181990,65c49a09d299486091b6586487679b87,9190cf28b35b425083bdb41e121822fe,clear,clear,clear,clear,clear,2017-06-20 22:35:40+00:00,clear,NaN,NaN,clear,clear,NaN,clear,NaN,"{'gender': 'Male', 'nationality': 'PRT', 'docu...",clear,2017-06-20


In [9]:
#doc_report = doc_report.drop('properties_v2', axis=1)

### Flatten properties

In [10]:
# Check data type of properties column 
doc_report['properties'].apply(type)

0         <class 'str'>
1         <class 'str'>
2         <class 'str'>
3         <class 'str'>
4         <class 'str'>
              ...      
176399    <class 'str'>
176400    <class 'str'>
176401    <class 'str'>
176402    <class 'str'>
176403    <class 'str'>
Name: properties, Length: 176404, dtype: object

In [11]:
# Handling strings that look like dicts
doc_report['properties'] = doc_report['properties'].apply(ast.literal_eval)

In [12]:
# Set the dtype explicitly for any empty entries before applying .apply(pd.Series) to avoid warning message
doc_report['properties'] = doc_report['properties'].apply(lambda x: x if isinstance(x, dict) else {})

# Flatten the dict column
doc_expanded = doc_report['properties'].apply(pd.Series)

In [13]:
# View flattened data
doc_expanded

,gender,nationality,document_type,date_of_expiry,issuing_country,issuing_date,issuing_state,document_version
0,Male,IRL,passport,2019-08-12,IRL,NaN,NaN,NaN
1,Female,NaN,driving_licence,2023-02-28,GBR,NaN,NaN,NaN
2,Male,ITA,passport,2018-06-09,ITA,NaN,NaN,NaN
3,Male,NaN,national_identity_card,NaN,FRA,2007-08,NaN,NaN
4,Male,POL,national_identity_card,2019-05-29,POL,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
176399,Female,CHN,passport,2027-04-23,CHN,NaN,NaN,NaN
176400,Female,NaN,driving_licence,2026-04-20,GBR,NaN,NaN,NaN
176401,Female,GBR,passport,2023-07-11,GBR,NaN,NaN,NaN
176402,Male,PRT,passport,2019-10-16,PRT,NaN,NaN,NaN


In [14]:
# Add user and attempt IDs to the dataset 
doc_expanded[['user_id','attempt_id']] = doc_report[['user_id','attempt_id']]

# Move ID columns to the front
cols = ['user_id','attempt_id'] + [col for col in doc_expanded.columns if col not in ['user_id','attempt_id']]
doc_expanded = doc_expanded[cols]

In [15]:
# See the dataset
doc_expanded

,user_id,attempt_id,gender,nationality,document_type,date_of_expiry,issuing_country,issuing_date,issuing_state,document_version
0,ab23fae164e34af0a1ad1423ce9fd9f0,050a0596de424fab83c433eaa18b3f8d,Male,IRL,passport,2019-08-12,IRL,NaN,NaN,NaN
1,15a84e8951254011b47412fa4e8f65b8,f69c1e5f45a64e50a26740b9bfb978b7,Female,NaN,driving_licence,2023-02-28,GBR,NaN,NaN,NaN
2,ffb82fda52b041e4b9af9cb4ef298c85,f9f84f3055714d8e8f7419dc984d1769,Male,ITA,passport,2018-06-09,ITA,NaN,NaN,NaN
3,bd4a8b3e3601427e88aa1d9eab9f4290,10a54a1ecf794404be959e030f11fef6,Male,NaN,national_identity_card,NaN,FRA,2007-08,NaN,NaN
4,f52ad1c7e69543a9940c3e7f8ed28a39,1f320d1d07de493292b7e0d5ebfb1cb9,Male,POL,national_identity_card,2019-05-29,POL,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
176399,c7f99ee763bf45d289019c6ac2cbd919,72a2cccc9e9942deb5274a16536bf2d0,Female,CHN,passport,2027-04-23,CHN,NaN,NaN,NaN
176400,8b47d72c363e4591861f523dd7487f20,8c83017191204a3887c2d47ca2d998ce,Female,NaN,driving_licence,2026-04-20,GBR,NaN,NaN,NaN
176401,3d16e02c245a4f1a8a76662ad933d5c4,bfea35bcb6a940118ca5816cd8ffcae7,Female,GBR,passport,2023-07-11,GBR,NaN,NaN,NaN
176402,65c49a09d299486091b6586487679b87,9190cf28b35b425083bdb41e121822fe,Male,PRT,passport,2019-10-16,PRT,NaN,NaN,NaN


In [16]:
# Nationality vs. issuing country


# Missing values
print(doc_expanded.isnull().sum())

user_id                  0
attempt_id               0
gender               56466
nationality          97932
document_type        26106
date_of_expiry       47289
issuing_country      26103
issuing_date        124611
issuing_state       175610
document_version    176342
dtype: int64


In [17]:
# Missing values
print(doc_report.isnull().sum())

number                                     0
user_id                                    0
attempt_id                                 0
result                                     0
visual_authenticity_result             26114
image_integrity_result                     1
face_detection_result                  26143
image_quality_result                       1
created_at                                 0
supported_document_result                504
conclusive_document_quality_result     81187
colour_picture_result                  81182
data_validation_result                 33430
data_consistency_result                84175
data_comparison_result                173856
police_record_result                   31847
compromised_document_result           130898
properties                                 0
sub_result                                 0
created_date                               0
dtype: int64


In [18]:
# Categorical column check: result
doc_report.value_counts('result')

result
clear       132402
consider     44002
Name: count, dtype: int64

### Facial similarity report

In [19]:
# View first few rows of the dataset
facial_report.head()

,number,user_id,result,face_comparison_result,created_at,facial_image_integrity_result,visual_authenticity_result,properties,attempt_id
0,0,ab23fae164e34af0a1ad1423ce9fd9f0,clear,clear,2017-06-20T23:12:58Z,clear,consider,{},050a0596de424fab83c433eaa18b3f8d
1,1,15a84e8951254011b47412fa4e8f65b8,clear,clear,2017-06-20T23:16:04Z,clear,clear,{},f69c1e5f45a64e50a26740b9bfb978b7
2,2,ffb82fda52b041e4b9af9cb4ef298c85,clear,clear,2017-06-20T17:59:49Z,clear,clear,{},f9f84f3055714d8e8f7419dc984d1769
3,3,bd4a8b3e3601427e88aa1d9eab9f4290,clear,clear,2017-06-20T17:59:39Z,clear,clear,{},10a54a1ecf794404be959e030f11fef6
4,4,f52ad1c7e69543a9940c3e7f8ed28a39,clear,clear,2017-06-20T18:08:09Z,clear,clear,{},1f320d1d07de493292b7e0d5ebfb1cb9


In [20]:
# Reorder columns to have IDs in the first two columns

# Get columns names 
cols = list(facial_report.columns)

# Move last column to second position
last_col = cols[-1]
new_order = [cols[0],cols[1],last_col] + cols[2:-1]

# Reorder the DF
facial_report = facial_report[new_order]

In [21]:
# Get a summary of the DataFrame (structure, data types, missing values)
facial_report.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 176404 entries, 0 to 176403
Data columns (total 9 columns):
 #   Column                         Non-Null Count   Dtype 
---  ------                         --------------   ----- 
 0   number                         176404 non-null  int64 
 1   user_id                        176404 non-null  object
 2   attempt_id                     176404 non-null  object
 3   result                         176403 non-null  object
 4   face_comparison_result         166007 non-null  object
 5   created_at                     176404 non-null  object
 6   facial_image_integrity_result  175941 non-null  object
 7   visual_authenticity_result     150290 non-null  object
 8   properties                     176404 non-null  object
dtypes: int64(1), object(8)
memory usage: 12.1+ MB


In [22]:
# Convert created_at to date format
facial_report['created_at'] = pd.to_datetime(facial_report['created_at'])

# Create 

In [23]:
# Check properties column: We suspect that they only contain {} and are not necessary
facial_report['properties_length'] = facial_report['properties'].str.len()

print(facial_report.groupby('properties_length').size())

# Express results as percentages
percentage_by_length = facial_report['properties_length'].value_counts(normalize=True) * 100
print(percentage_by_length)

properties_length
2     172921
14       324
15      3159
dtype: int64
properties_length
2     98.025555
15     1.790776
14     0.183669
Name: proportion, dtype: float64


In [24]:
# Check information in propertie s
facial_report['properties'].value_counts(normalize=True) * 100

properties
{}                 98.025555
{'score': 0.69}     0.095803
{'score': 0.64}     0.092402
{'score': 0.63}     0.090134
{'score': 0.66}     0.090134
                     ...    
{'score': 0.1}      0.000567
{'score': 0.14}     0.000567
{'score': 0.88}     0.000567
{'score': 0.19}     0.000567
{'score': 0.21}     0.000567
Name: proportion, Length: 80, dtype: float64

In [25]:
# 
facial_report.loc[facial_report['properties'] != '{}','properties']

710       {'score': 0.69}
712       {'score': 0.67}
732       {'score': 0.64}
763       {'score': 0.51}
765       {'score': 0.56}
               ...       
172439    {'score': 0.66}
173187    {'score': 0.82}
173544    {'score': 0.67}
173561     {'score': 0.7}
174884    {'score': 0.77}
Name: properties, Length: 3483, dtype: object

In [26]:
facial_report['properties'].apply(type)

0         <class 'str'>
1         <class 'str'>
2         <class 'str'>
3         <class 'str'>
4         <class 'str'>
              ...      
176399    <class 'str'>
176400    <class 'str'>
176401    <class 'str'>
176402    <class 'str'>
176403    <class 'str'>
Name: properties, Length: 176404, dtype: object

In [27]:
# Handling strings that look like dicts
facial_report['properties_v2'] = facial_report['properties'].apply(ast.literal_eval)

In [28]:
# Flatten the dict column
facial_report_expanded = facial_report['properties_v2'].apply(pd.Series, dtype='str')

In [29]:
# Add user and attempt IDs to the dataset 
facial_report_expanded[['user_id','attempt_id']] = facial_report[['user_id','attempt_id']]

# Move ID columns to the front
cols = ['user_id','attempt_id'] + [col for col in facial_report_expanded.columns if col not in ['user_id','attempt_id']]
facial_report_expanded = facial_report_expanded[cols]

In [30]:
facial_report_expanded

,user_id,attempt_id,score
0,ab23fae164e34af0a1ad1423ce9fd9f0,050a0596de424fab83c433eaa18b3f8d,NaN
1,15a84e8951254011b47412fa4e8f65b8,f69c1e5f45a64e50a26740b9bfb978b7,NaN
2,ffb82fda52b041e4b9af9cb4ef298c85,f9f84f3055714d8e8f7419dc984d1769,NaN
3,bd4a8b3e3601427e88aa1d9eab9f4290,10a54a1ecf794404be959e030f11fef6,NaN
4,f52ad1c7e69543a9940c3e7f8ed28a39,1f320d1d07de493292b7e0d5ebfb1cb9,NaN
...,...,...,...
176399,c7f99ee763bf45d289019c6ac2cbd919,72a2cccc9e9942deb5274a16536bf2d0,NaN
176400,8b47d72c363e4591861f523dd7487f20,8c83017191204a3887c2d47ca2d998ce,NaN
176401,3d16e02c245a4f1a8a76662ad933d5c4,bfea35bcb6a940118ca5816cd8ffcae7,NaN
176402,65c49a09d299486091b6586487679b87,9190cf28b35b425083bdb41e121822fe,NaN


### Data cleaning

### Data analysis

In [31]:
doc_report.columns

Index(['number', 'user_id', 'attempt_id', 'result',
       'visual_authenticity_result', 'image_integrity_result',
       'face_detection_result', 'image_quality_result', 'created_at',
       'supported_document_result', 'conclusive_document_quality_result',
       'colour_picture_result', 'data_validation_result',
       'data_consistency_result', 'data_comparison_result',
       'police_record_result', 'compromised_document_result', 'properties',
       'sub_result', 'created_date'],
      dtype='object')

In [32]:
doc_report_subset = doc_report.drop(['number','properties'],axis=1)

In [33]:
doc_report_subset = doc_report_subset.rename(columns ={
    'result':'doc_result',
    'created_at':'doc_created_at',
    'created_date':'doc_created_date',
    'sub_result':'doc_sub_result'
})

In [34]:
facial_report.columns

Index(['number', 'user_id', 'attempt_id', 'result', 'face_comparison_result',
       'created_at', 'facial_image_integrity_result',
       'visual_authenticity_result', 'properties', 'properties_length',
       'properties_v2'],
      dtype='object')

In [35]:
facial_report_subset = facial_report.drop(['properties','properties_length','properties_v2'],axis=1)

In [36]:
facial_report_subset = facial_report_subset.rename(columns = {
    'result':'facial_result',
    'created_at':'facial_created_at',
    'visual_authenticity_result':'facial_visal'
})

In [37]:
facial_report_subset

,number,user_id,attempt_id,facial_result,face_comparison_result,facial_created_at,facial_image_integrity_result,visual_authenticity_result
0,0,ab23fae164e34af0a1ad1423ce9fd9f0,050a0596de424fab83c433eaa18b3f8d,clear,clear,2017-06-20 23:12:58+00:00,clear,consider
1,1,15a84e8951254011b47412fa4e8f65b8,f69c1e5f45a64e50a26740b9bfb978b7,clear,clear,2017-06-20 23:16:04+00:00,clear,clear
2,2,ffb82fda52b041e4b9af9cb4ef298c85,f9f84f3055714d8e8f7419dc984d1769,clear,clear,2017-06-20 17:59:49+00:00,clear,clear
3,3,bd4a8b3e3601427e88aa1d9eab9f4290,10a54a1ecf794404be959e030f11fef6,clear,clear,2017-06-20 17:59:39+00:00,clear,clear
4,4,f52ad1c7e69543a9940c3e7f8ed28a39,1f320d1d07de493292b7e0d5ebfb1cb9,clear,clear,2017-06-20 18:08:09+00:00,clear,clear
...,...,...,...,...,...,...,...,...
176399,181987,c7f99ee763bf45d289019c6ac2cbd919,72a2cccc9e9942deb5274a16536bf2d0,clear,clear,2017-06-20 22:25:53+00:00,clear,clear
176400,181988,8b47d72c363e4591861f523dd7487f20,8c83017191204a3887c2d47ca2d998ce,clear,clear,2017-06-20 22:27:40+00:00,clear,clear
176401,181989,3d16e02c245a4f1a8a76662ad933d5c4,bfea35bcb6a940118ca5816cd8ffcae7,clear,clear,2017-06-20 22:25:59+00:00,clear,clear
176402,181990,65c49a09d299486091b6586487679b87,9190cf28b35b425083bdb41e121822fe,clear,clear,2017-06-20 22:35:41+00:00,clear,clear


In [45]:
combined_check_results = facial_report_subset.merge(doc_report_subset, on=['user_id','attempt_id'],how='left')

In [65]:
combined_check_results['visual_authenticity_result_x'] = combined_check_results['visual_authenticity_result_x'].str.strip()
combined_check_results['visual_authenticity_result_y'] = combined_check_results['visual_authenticity_result_y'].str.strip()

In [68]:
# Check if visual authenticity columns are the same
(combined_check_results['visual_authenticity_result_x'] == combined_check_results['visual_authenticity_result_y']).all()

False

In [67]:
combined_check_results['visual_authenticity_result_x'] != combined_check_results['visual_authenticity_result_y']

0         False
1         False
2         False
3         False
4         False
          ...  
176399    False
176400    False
176401    False
176402    False
176403    False
Length: 176404, dtype: bool

In [69]:
test = combined_check_results[['visual_authenticity_result_x','visual_authenticity_result_y']]

In [70]:
(test['visual_authenticity_result_x'] == test['visual_authenticity_result_y'])

0         True
1         True
2         True
3         True
4         True
          ... 
176399    True
176400    True
176401    True
176402    True
176403    True
Length: 176404, dtype: bool

In [71]:
test['visual_authenticity_result_x'].equals(test['visual_authenticity_result_y'])

False

In [73]:
test[test['visual_authenticity_result_x'] != test['visual_authenticity_result_y']]

,visual_authenticity_result_x,visual_authenticity_result_y
711,NaN,NaN
719,NaN,NaN
721,NaN,NaN
728,NaN,NaN
730,NaN,NaN
...,...,...
174434,NaN,NaN
175348,NaN,NaN
176014,NaN,NaN
176126,NaN,NaN


In [39]:
combined_check_results.head()

,number_x,user_id,attempt_id,result_x,face_comparison_result,created_at_x,facial_image_integrity_result,visual_authenticity_result_x,properties_x,properties_length,...,conclusive_document_quality_result,colour_picture_result,data_validation_result,data_consistency_result,data_comparison_result,police_record_result,compromised_document_result,properties_y,sub_result,created_date
0,0,ab23fae164e34af0a1ad1423ce9fd9f0,050a0596de424fab83c433eaa18b3f8d,clear,clear,2017-06-20 23:12:58+00:00,clear,consider,{},2,...,NaN,NaN,clear,clear,NaN,clear,NaN,"{'gender': 'Male', 'nationality': 'IRL', 'docu...",caution,2017-06-20
1,1,15a84e8951254011b47412fa4e8f65b8,f69c1e5f45a64e50a26740b9bfb978b7,clear,clear,2017-06-20 23:16:04+00:00,clear,clear,{},2,...,NaN,NaN,clear,NaN,NaN,clear,NaN,"{'gender': 'Female', 'document_type': 'driving...",clear,2017-06-20
2,2,ffb82fda52b041e4b9af9cb4ef298c85,f9f84f3055714d8e8f7419dc984d1769,clear,clear,2017-06-20 17:59:49+00:00,clear,clear,{},2,...,NaN,NaN,clear,clear,NaN,clear,NaN,"{'gender': 'Male', 'nationality': 'ITA', 'docu...",clear,2017-06-20
3,3,bd4a8b3e3601427e88aa1d9eab9f4290,10a54a1ecf794404be959e030f11fef6,clear,clear,2017-06-20 17:59:39+00:00,clear,clear,{},2,...,NaN,NaN,clear,clear,NaN,clear,NaN,"{'gender': 'Male', 'issuing_date': '2007-08', ...",clear,2017-06-20
4,4,f52ad1c7e69543a9940c3e7f8ed28a39,1f320d1d07de493292b7e0d5ebfb1cb9,clear,clear,2017-06-20 18:08:09+00:00,clear,clear,{},2,...,NaN,NaN,clear,clear,NaN,clear,NaN,"{'gender': 'Male', 'nationality': 'POL', 'docu...",clear,2017-06-20


In [40]:
# Check if properties columns match exactly 
matches = (combined_check_results['properties_x'] == combined_check_results['properties_y']).sum()
print(matches)

0
